# Multi-Target QSAR Predictor API (EGFR & BRAF)

In [1]:
import sys
get_ipython().system(f'{sys.executable} -m pip install chembl-webresource-client')

In [1]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

target_api = new_client.target
braf_results = target_api.search("BRAF")

# Print all results so we pick the RIGHT one
for r in braf_results:
    print(r["target_chembl_id"], "-", r["pref_name"],
          "-", r["organism"])

CHEMBL2331061 - Serine/threonine-protein kinase B-raf - Mus musculus
CHEMBL4106189 - BRAF/CRAF - Homo sapiens
CHEMBL5145 - Serine/threonine-protein kinase B-raf - Homo sapiens
CHEMBL4523687 - Protein cereblon/Serine/threonine-protein kinase B-raf - Homo sapiens
CHEMBL5291967 - von Hippel-Lindau disease tumor suppressor/Serine/threonine-protein kinase B-raf - Homo sapiens
CHEMBL3883317 - B-raf/RAF proto-oncogene serine/threonine-protein kinase - Homo sapiens
CHEMBL3559685 - RAF serine/threonine protein kinase - Homo sapiens


# Fetching real bioactivity data once we confirm the ID:

In [2]:
activity_api = new_client.activity

activities = activity_api.filter(
    target_chembl_id="CHEMBL5145",   # confirm this matches your Step 2 output
    standard_type="IC50",
    relation="=",
    assay_type="B"
).only(["molecule_chembl_id", "canonical_smiles", "standard_value", "standard_units"])

df_raw = pd.DataFrame(list(activities))
print(f"Fetched {len(df_raw)} raw activity records")
print(df_raw.head())

Fetched 9047 raw activity records
                                    canonical_smiles molecule_chembl_id  \
0        CC(C)(C)c1nc(-c2ccc(F)cc2)c(-c2ccncc2)[nH]1       CHEMBL371694   
1        OCc1cccc(-c2[nH]c(-c3ccccc3)nc2-c2ccncc2)c1       CHEMBL200259   
2         Oc1ccc(-c2[nH]c(-c3ccccc3)nc2-c2ccncc2)cc1        CHEMBL68215   
3  CC(C)(CNC(=O)Nc1ccc(Cl)cc1)c1nc(-c2ccc(Cl)c(O)...       CHEMBL200863   
4  CC(C)(CNS(C)(=O)=O)c1nc(-c2ccc(Cl)c(O)c2)c(-c2...       CHEMBL381447   

  standard_units standard_value units   value  
0             nM          900.0    nM   900.0  
1             nM         1585.0    nM  1585.0  
2             nM          339.0    nM   339.0  
3             nM           13.0    nM    13.0  
4             nM           14.0    nM    14.0  


## Step 1 — Clean the raw data:

In [3]:
import numpy as np

# Clean duplicates and nulls
df_raw = df_raw.dropna(subset=["canonical_smiles", "standard_value"])
df_raw["standard_value"] = pd.to_numeric(df_raw["standard_value"], errors="coerce")
df_raw = df_raw.dropna(subset=["standard_value"])
df_raw = df_raw[df_raw["standard_value"] > 0]

# Remove duplicate molecules (keep first occurrence)
df_raw = df_raw.drop_duplicates(subset=["canonical_smiles"])

print(f"After cleaning: {len(df_raw)} unique compounds")
print(f"\nIC50 distribution (nM):")
print(df_raw["standard_value"].describe())

After cleaning: 5386 unique compounds

IC50 distribution (nM):
count    5.386000e+03
mean     1.261242e+04
std      4.840282e+05
min      2.000000e-02
25%      3.700000e+00
50%      2.100000e+01
75%      3.000000e+02
max      2.500034e+07
Name: standard_value, dtype: float64


# Step 1: Establish the Classification Threshold

This cell creates the active column by throwing away the ambiguous "grey area" compounds (between 1,000 nM and 10,000 nM). It ensures our model only learns from high-confidence chemistry.

In [7]:
# Create a fresh copy to avoid Pandas warnings
df = df_raw.copy()

# Filter out the ambiguous middle ground
df = df[(df["standard_value"] <= 1000) | (df["standard_value"] >= 10000)].copy()

# Create binary labels: 1 for Active (<= 1000 nM), 0 for Inactive (>= 10000 nM)
df["active"] = (df["standard_value"] <= 1000).astype(int)

# Optional: Calculate pIC50 for future reference
df["pIC50"] = df["standard_value"].apply(lambda x: round(9 - np.log10(x), 2))

print(f"Active compounds:   {df['active'].sum()}")
print(f"Inactive compounds: {(df['active']==0).sum()}")
print(f"Total dataset:      {len(df)}")

Active compounds:   4579
Inactive compounds: 261
Total dataset:      4840


# Step 2: Feature Engineering (RDKit)

This cell iterates through our clean dataframe and converts every single SMILES string into the 1030-length hybrid feature vector (6 descriptors + 1024 Morgan fingerprint bits).

Crucially, we do this before the Train/Test split so we don't accidentally misalign our dataset if RDKit fails to parse a specific SMILES string.

In [8]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

def compute_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # 6 1D Descriptors
    desc = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
    ]
    
    # 1024 2D Morgan Fingerprint bits
    fp = list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024))
    
    return desc + fp

print("Extracting chemical features... this will take a minute or two.")
features_list = []
valid_idx = []

for i, smiles in enumerate(df["canonical_smiles"]):
    f = compute_features(smiles)
    if f is not None:
        features_list.append(f)
        valid_idx.append(i)  # Keep track of which molecules successfully parsed
    
    if (i + 1) % 1000 == 0:
        print(f"Processed {i+1}/{len(df)}...")

# Create the final machine learning matrices
X = np.array(features_list)
y = df["active"].iloc[valid_idx].values

print(f"\nFinal feature matrix X: {X.shape}")
print(f"Final label array y:    {y.shape}")

Extracting chemical features... this will take a minute or two.
Processed 1000/4840...
Processed 2000/4840...
Processed 3000/4840...
Processed 4000/4840...

Final feature matrix X: (4840, 1030)
Final label array y:    (4840,)


# Step-4 Model Building

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

print("1. Splitting data (ensuring no data leakage)...")
# Chronology matters: Split the final feature arrays AFTER RDKit extraction
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("2. Scaling features...")
# Initialize the scaler
scaler = StandardScaler()
# Fit only on training data, then transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("3. Training the Random Forest (with class_weight='balanced')...")
# The 'Careful Inspector' model
rf_model = RandomForestClassifier(
    n_estimators=200, 
    class_weight="balanced", 
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

print("4. Evaluating the final model...")
predictions = rf_model.predict(X_test_scaled)
probabilities = rf_model.predict_proba(X_test_scaled)[:, 1]

print("\n--- Final Exam Results ---")
print(classification_report(y_test, predictions, target_names=["Inactive", "Active"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, probabilities):.3f}")

# Extract exact numbers for the rare Inactives
cm = confusion_matrix(y_test, predictions)
n_inactive_total = (y_test == 0).sum()
n_inactive_correct = cm[0][0]
print(f"\nOf {n_inactive_total} truly inactive test compounds, "
      f"the model correctly identified: {n_inactive_correct} "
      f"({n_inactive_correct/n_inactive_total:.1%})")

print("\n5. Extracting Applicability Domain data...")
# Extract ONLY the 1024 Morgan Fingerprint bits from the training set (skipping the 6 descriptors)
braf_train_fp = X_train[:, 6:]

print("6. Saving all assets to disk for FastAPI deployment...")
# Save the model, the scaler, and the raw fingerprints
joblib.dump(rf_model, "braf_model.pkl")
joblib.dump(scaler, "braf_scaler.pkl")
joblib.dump(braf_train_fp, "braf_train_fp.pkl")

print("\n✅ Success! braf_model.pkl, braf_scaler.pkl, and braf_train_fp.pkl are ready for deployment.")

1. Splitting data (ensuring no data leakage)...
2. Scaling features...
3. Training the Random Forest (with class_weight='balanced')...
4. Evaluating the final model...

--- Final Exam Results ---
              precision    recall  f1-score   support

    Inactive       0.90      0.71      0.80        52
      Active       0.98      1.00      0.99       916

    accuracy                           0.98       968
   macro avg       0.94      0.85      0.89       968
weighted avg       0.98      0.98      0.98       968

ROC-AUC Score: 0.979

Of 52 truly inactive test compounds, the model correctly identified: 37 (71.2%)

5. Extracting Applicability Domain data...
6. Saving all assets to disk for FastAPI deployment...

✅ Success! braf_model.pkl, braf_scaler.pkl, and braf_train_fp.pkl are ready for deployment.


# Prerequisite: Saving the Training Fingerprints

Before you run the API, your server needs to know what the training data actually looked like so the "Bouncer" can compare new molecules to it. Run this quick snippet in your Jupyter Notebook to save the raw fingerprints:

In [10]:
import joblib

# Extract just the 1024-bit fingerprints (skipping the 6 1D descriptors)
braf_X_train_fp = X_train[:, 6:]
joblib.dump(braf_X_train_fp, "braf_train_fp.pkl")

# (Do the exact same thing for your EGFR notebook and save as "egfr_train_fp.pkl")

['braf_train_fp.pkl']

# The Complete main.py Pipeline

This is the master file for our FastAPI application.

In [15]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. Initialize API ---
app = FastAPI(
    title="Multi-Target QSAR Predictor",
    description="Machine Learning API for predicting EGFR and BRAF kinase inhibition.",
    version="2.0"
)

# --- 2. Load Models Globally ---
try:
    # New BRAF Assets
    braf_rf_model = joblib.load("braf_model.pkl")
    braf_scaler = joblib.load("braf_scaler.pkl")
    braf_train_fp = joblib.load("braf_train_fp.pkl")
    
    # Old EGFR Asset (Classic Pipeline)
    egfr_rf_model = joblib.load("qsar_model.pkl")
except FileNotFoundError as e:
    raise RuntimeError(f"Startup Failed: Missing asset file. {e}")

# --- 3. Strict Payload Validation ---
class MoleculeRequest(BaseModel):
    smiles: str

# --- 4. Core Cheminformatics Functions ---
def compute_braf_features(smiles: str):
    """Generates the 1030-feature vector for the new BRAF pipeline."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    desc = [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol)
    ]
    fp = list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024))
    return np.array(desc + fp).reshape(1, -1)

def compute_egfr_features(smiles: str):
    """Generates ONLY the 1024-bit fingerprint for the classic EGFR pipeline."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024))
    return np.array(fp).reshape(1, -1)

def is_in_domain(query_features: np.ndarray, reference_fingerprints: np.ndarray, threshold: float = 0.3) -> dict:
    """The 'Bouncer' for the BRAF model."""
    query_fp = query_features[:, 6:] # Skip the 6 descriptors
    similarities = cosine_similarity(query_fp, reference_fingerprints)
    max_sim = float(similarities.max())
    return {"is_valid": max_sim >= threshold, "max_similarity": round(max_sim, 3)}

# --- 5. API Endpoints ---

@app.post("/predict_braf")
def predict_braf(request: MoleculeRequest):
    features = compute_braf_features(request.smiles)
    if features is None:
        raise HTTPException(status_code=400, detail="Invalid SMILES string provided.")
    
    ad_check = is_in_domain(features, braf_train_fp)
    if not ad_check["is_valid"]:
        return {
            "target": "BRAF",
            "smiles": request.smiles,
            "prediction": "Out of Domain",
            "warning": "Molecule lacks structural similarity to training data.",
            "max_training_similarity": ad_check["max_similarity"]
        }
    
    features_scaled = braf_scaler.transform(features)
    prediction = braf_rf_model.predict(features_scaled)[0]
    probability = braf_rf_model.predict_proba(features_scaled)[0][1]
    
    return {
        "target": "BRAF",
        "smiles": request.smiles,
        "prediction": "Active" if prediction == 1 else "Inactive",
        "max_training_similarity": ad_check["max_similarity"],
        "active_probability": round(float(probability), 3)
    }

@app.post("/predict_egfr")
def predict_egfr(request: MoleculeRequest):
    # Uses the classic, simplified feature extractor
    features = compute_egfr_features(request.smiles)
    if features is None:
        raise HTTPException(status_code=400, detail="Invalid SMILES string provided.")
    
    # Predicts directly without scaling or checking Applicability Domain
    prediction = egfr_rf_model.predict(features)[0]
    probability = egfr_rf_model.predict_proba(features)[0][1]
    
    return {
        "target": "EGFR",
        "smiles": request.smiles,
        "prediction": "Active" if prediction == 1 else "Inactive",
        "active_probability": round(float(probability), 3),
        "note": "Legacy model (No Applicability Domain filter applied)"
    }